# 量化：把模型塞进去

一个70B参数的模型使用FP16需要140GB，需要两张A100才能存下权重。如果量化到FP8，一张80GB的GPU就可以了。如果量化到INT4，一个MacBook也能塞下。

## 问题描述

使用16bits存储参数很浪费，神经网络中的大部分权重都聚集在0附近。FP16的数值范围为0.000000059到65504大部分都没有使用。如果你测量一下Llama3中370B权重参数的分布，95%都落在-0.1到+0.1之间。

量化将高精度数值替换成低精度数值。FP16替换成FP8使显存减半，替换成INT4只要四分之一，如果只用两个bit量化，能降到八分之一，不过要小心过于激进的量化对于某些任务不稳定。

代价很明显。你移除的每个bit都会导致信息丢失。答案在于你究竟在哪些地方丢失的了多少精度。一个量化做的好的INT4模型能够保留原始模型95-99%的基准性能，量化做的不好，可能会毁了整个模型。

## 基本概念

### 数值格式：每个bit在干嘛

每个浮点数都包含3个部分，符号、指数部分、消暑部分。
|格式|符号（Bit）|指数（Bits）|小数（Bits）|合计（Bits）|数值范围|
|---|---|---|---|---|---|
|FP32|1|8|23|32|1.2e-38~3.4e38|
|FP16|1|5|10|16|5.9e-8~65004|
|BF16|1|8|7|16|>5.9e-8~3.4e38|
|FP8(E4M3)|1|4|3|8|用于推理|
|FP8(E5M2)|1|5|2|8|用于梯度|
|INT8|1|7 values||8|-128～127|
|INT4|1|4 values||4|-16~15|

## 量化工作的核心

对于浮点数值的张量，取一个缩放因子，乘上去，然后近似到最近的整数，最后存这个整数和缩放系数。

量化：
```
scale = max(abs(tensor)) / max_int_value
quantized = round(tensor/ scale)
```
去量化：
```
reconstructed = quantized * scale
```

损失来自round近似，最高可达scale/2，整个层的误差取决于你有多少权重，以及这些权重对扰动有多敏感。

### 逐张量 VS 逐通道量化

逐张量量化对整个权重矩阵使用同一个缩放因子。简单但容易出问题，如果某一列的数据明显大于其他列，对于数值较小的哪些列容易丢失更多的精度。逐通道量化则是每列使用一个缩放因子，你需要存储N个缩放因子而不是一个，但是通常有更好的表现。

### 非对称量化
```
quantized = round(tensor / scale) + zero_point
```
用来处理分布没有集中在0的情况，比如ReLU激活函数，没有负值。使用对称量化会损失一半的数值范围。所以说非对称量化就是把[min, max]映射到可使用的数值范围。

### 敏感性

模型中不是所有的东西都平等地能够忍受量化，有一条清晰的轨迹。

- 权重最鲁棒。训练过程中变化很小，且高斯分布在0附近。
- 激活函数一般敏感。因为有outliers出现，数值范围更大。
- KV cache高度敏感。随上下文长度增加，误差会累积。
- 注意力分数最敏感。因为过了一层softmax函数，指数操作会放大小的数值变化。

### PTQ VS QAT

- 训练后量化(Post-Training Quantization, PTQ)，对一个一训练的模型做量化，没有重新训练的过程。

- 量化感知训练（Quantization-Aware Training），在训练前向传播的时候插入伪的量化操作，所以模型学会了把权重放在近似误差小的地方。梯度通过直通估计器流过伪量化，假装该操作的梯度是1。

### 量化度量

你怎么知道量化后的模型仍然保持健康？

- 困惑度。最通用的度量准则，看量化后模型相对于原模型的困惑度变化。
- 任务特定基准。量化前后模型都跑基准测试。
- 输出对比。量化前后同一提示词生成的回复，加上人工对比或者大模型对比。
- 延迟和吞吐。量化的目标就是为了让模型更快更便宜，所以需要作为基准。

# 动手编码

In [2]:
import numpy as np

def float_to_fp32_bits(value):
    bits = np.float32(value).view(np.int32)
    sign = (bits >> 31) & 0x1
    exponent = (bits >> 23) & 0xFF
    mantissa = bits & 0x7FFFFF
    return {
        "sign": int(sign),
        "exponent": int(exponent),
        "mantissa": int(mantissa),
        "exponent_bits": format(int(exponent), "08b"),
        "mantissa_bits": format(int(mantissa), "023b"),
        "value": float(value),
        "actual_exponent": exponent - 127,
    }


print(float_to_fp32_bits(1.1))

{'sign': 0, 'exponent': 127, 'mantissa': 838861, 'exponent_bits': '01111111', 'mantissa_bits': '00011001100110011001101', 'value': 1.1, 'actual_exponent': np.int32(0)}


In [ ]:
def quantize_symmetric(tensor, num_bits=8):
    qmin = -2**(num_bits - 1)
    qmax = 2**(num_bits - 1) - 1

    abs_max = np.max(np.abs(tensor))
    if abs_max == 0:
        return np.zeros_like(tensor, dtype=np.int32), 1.0

    scale = abs_max / qmax

    quantized = np.clip(
        np.round((tensor / scale), qmin, qmax).astype(np.int32)
    )

    return quantized, float(scale)

def dequantize_symmetric(quantized, scale):
    return quantized.astype(np.float64) * scale    

In [ ]:
def quantization_error(original, reconstructed):
    diff = original - reconstructed
    mse = float(np.mean(diff ** 2))
    rmse = float(np.sqrt(mse))

    max_error = float(np.max(np.abs(diff)))
    signal_power = float(np.mean(original ** 2))
    snr_db = 10 * np.log10(signal_power / max(mse, 1e-20))

    orig_flat = original.flatten()
    rec_flat = reconstructed.flatten()
    norm_orig = np.linalg.norm(orig_flat)
    norm_rec = np.linalg.norm(rec_flat)

    cos_sim = np.dot(orig_flat, rec_flat) / (norm_orig * norm_rec)
    
    return {
        "mse": mse,
        "rmse": rmse,
        "snr_db": snr_db,
        "cos_sim": cos_sim,
        "max_error": max_error,
    }
    
    